In [1]:
import pandas as pd
import numpy as np

# Загружаем данные
df = pd.read_csv('../data/raw/games.csv')

# Смотрим размер датасета
print(f'Строк: {df.shape[0]}')
print(f'Колонок: {df.shape[1]}')

Строк: 125855
Колонок: 39


## Датасет Steam

Датасет содержит **125 855 игр** с 39 атрибутами каждая: цена, отзывы, жанры, разработчики, время в игре и платформы.

In [2]:
# Смотрим названия всех колонок
print(df.columns.tolist())

['AppID', 'Name', 'Release date', 'Estimated owners', 'Peak CCU', 'Required age', 'Price', 'DiscountDLC count', 'About the game', 'Supported languages', 'Full audio languages', 'Reviews', 'Header image', 'Website', 'Support url', 'Support email', 'Windows', 'Mac', 'Linux', 'Metacritic score', 'Metacritic url', 'User score', 'Positive', 'Negative', 'Score rank', 'Achievements', 'Recommendations', 'Notes', 'Average playtime forever', 'Average playtime two weeks', 'Median playtime forever', 'Median playtime two weeks', 'Developers', 'Publishers', 'Categories', 'Genres', 'Tags', 'Screenshots', 'Movies']


In [3]:
df.head()


,AppID,Name,Release date,Estimated owners,Peak CCU,Required age,Price,DiscountDLC count,About the game,Supported languages,...,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Developers,Publishers,Categories,Genres,Tags,Screenshots,Movies
2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0 - 0,0,0,0.00,0,0,NaN,[],...,0,0,0,NaN,NaN,NaN,NaN,NaN,https://shared.akamai.steamstatic.com/store_it...,NaN
496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0 - 20000,0,0,5.24,65,0,"Springtime, April: when the cherry trees come ...",['English'],...,0,8,0,minori,MangaGamer,"Single-player,Steam Trading Cards,Steam Cloud,...",Adventure,"Adventure,Visual Novel,Anime,Cute",https://shared.akamai.steamstatic.com/store_it...,NaN
1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0 - 20000,0,0,4.99,0,0,"Immerse yourself in the most beloved, mystical...","['English', 'French', 'German', 'Russian']",...,0,0,0,Somer Games,8floor,"Single-player,Family Sharing",Casual,"Casual,Card Game,Solitaire,Puzzle,Hidden Objec...",https://shared.akamai.steamstatic.com/store_it...,NaN
3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",0 - 20000,1,0,8.99,0,1,"synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...",['Korean'],...,0,0,0,유진게임즈,유진게임즈,"Single-player,Steam Achievements,Family Sharing","Casual,Indie,Simulation",NaN,https://shared.akamai.steamstatic.com/store_it...,NaN
3631080,Maze Quest VR,"Apr 24, 2025",0 - 20000,0,0,4.99,0,0,Its not just a Maze; its a Quest! Enter the ca...,['English'],...,0,0,0,Reality Expanded LLC,Reality Expanded LLC,"Single-player,VR Only,Steam Leaderboards,Famil...","Action,Early Access",NaN,https://shared.akamai.steamstatic.com/store_it...,NaN


Колонка `DiscountDLC count` — склеенный заголовок вместо двух отдельных (`Discount` и `DLC count`). 
Это дефект исходного CSV: потеряна запятая в заголовке, из-за чего все последующие колонки сдвинуты на одну позицию. В ETL-пайплайне дефект исправлен явным заданием имён колонок.

In [4]:
df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 125855 entries, 2539430 to 4247310
Data columns (total 39 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   AppID                       125854 non-null  object 
 1   Name                        125855 non-null  object 
 2   Release date                125855 non-null  object 
 3   Estimated owners            125855 non-null  int64  
 4   Peak CCU                    125855 non-null  int64  
 5   Required age                125855 non-null  float64
 6   Price                       125855 non-null  int64  
 7   DiscountDLC count           125855 non-null  int64  
 8   About the game              117393 non-null  object 
 9   Supported languages         125855 non-null  object 
 10  Full audio languages        125855 non-null  object 
 11  Reviews                     12181 non-null   object 
 12  Header image                125774 non-null  object 
 13  Website     

Большинство числовых колонок заполнены полностью. Проблемные поля — текстовые метаданные (`Reviews`, `Notes`, `Website`).

In [5]:
# Считаем пропуски в процентах
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_df = pd.DataFrame({
    'Пропусков': missing,
    'Процент': missing_pct
}).sort_values('Процент', ascending=False)

print(missing_df[missing_df['Пропусков'] > 0])

                Пропусков  Процент
Movies             125855    100.0
Score rank         125815    100.0
Metacritic url     121597     96.6
Reviews            113674     90.3
Notes              102545     81.5
Website             75335     59.9
Support url         70560     56.1
Tags                42502     33.8
Support email       22630     18.0
Categories           8966      7.1
Publishers           8922      7.1
Genres               8423      6.7
About the game       8462      6.7
Developers           8449      6.7
Screenshots          6027      4.8
Header image           81      0.1
AppID                   1      0.0


## Пропуски в данных

`Movies` и `Score rank` — фактически пустые колонки (100% и 99.9% пропусков), исключаем из анализа.

**Ключевое наблюдение:** 8 423 игры (6.7%) не имеют жанра. Это плейтесты, демо-версии и служебные страницы Steam — они попали в каталог но не являются полноценными играми. При анализе жанров эти записи будут исключены.

In [6]:
# Выбираем нужные колонки и смотрим на них
cols_to_keep = [
    'AppID', 'Name', 'Release date', 'Estimated owners',
    'Peak CCU', 'Price', 'Positive', 'Negative',
    'Average playtime forever', 'Median playtime forever',
    'Developers', 'Publishers', 'Categories', 'Genres', 'Tags',
    'Windows', 'Mac', 'Linux', 'Required age'
]

df_clean = df[cols_to_keep].copy()
print(f'Оставляем колонок: {len(cols_to_keep)}')
print(f'Строк: {len(df_clean)}')
df_clean.head(3)


Оставляем колонок: 19
Строк: 125855


,AppID,Name,Release date,Estimated owners,Peak CCU,Price,Positive,Negative,Average playtime forever,Median playtime forever,Developers,Publishers,Categories,Genres,Tags,Windows,Mac,Linux,Required age
2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0 - 0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,True,False,False,0.00
496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0 - 20000,0,0,65,252,3,8,8,minori,MangaGamer,"Single-player,Steam Trading Cards,Steam Cloud,...",Adventure,"Adventure,Visual Novel,Anime,Cute",True,False,False,5.24
1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0 - 20000,0,0,0,21,3,0,0,Somer Games,8floor,"Single-player,Family Sharing",Casual,"Casual,Card Game,Solitaire,Puzzle,Hidden Objec...",True,True,False,4.99


Оставляем 19 содержательных колонок из 39. Убираем технические метаданные (URL, скриншоты, описания) — они не нужны для количественного анализа.